# Ohjeet

1. Täytä vuorotoive-Excel:
   * Ensimmäinen välilehti **Pyhäpäivät**: merkitse pyhäpäivät x:llä. Neljänteen sarakkeeseen voi merkitä X-vuorojen lukumäärän päivälle (esim. `2`). Tyhjä solu = oletus (maanantaisin 1 X-vuoro, muina päivinä ei yhtään). `0` poistaa päivän X-vuorot.
   * Muut välilehdet: yksi välilehti per työntekijä, sarake **Vuorotoive** jokaiselle päivälle.
2. Lataa Excel istuntoon: *Files > Upload to session storage*.
3. Tarkista, että `INPUT_FILE` (kohta 1 alla) on sama kuin Excelin nimi.
4. Paina *Run all*.
5. Lataa tulos `vuorot_ratkaisu.xlsx` kohdasta *Files*. Vuorolistan alla punaisella on täyttämättömät vuorot, jotka vuorovastaava täyttää käsin.

**Vuorotoiveiden kirjoittaminen**

| Solun teksti | Tulkinta |
| --- | --- |
| `Kaikki käy` | Kaikki päivän vuorot käyvät |
| `Vapaatoive` (myös esim. `vapaatoive, luento`) | Ei vuoroa sinä päivänä |
| Vuorokoodit, esim. `I2 ja I3 käy` | Vain mainitut vuorot käyvät |
| Tyhjä solu tai muu teksti (esim. `iltavuoro käy`) | Ei vuoroa sinä päivänä. Nämä solut tulostetaan listana, jotta vuorovastaava näkee ne. |

**Huom.**
* Jakson ensimmäisen päivän pitää olla maanantai ja viimeisen sunnuntai (kokonaisia viikkoja).
* Joskus Colab pyytää käynnistämään istunnon uudelleen (*Restart session*). Paina sen jälkeen uudelleen *Run all*.

# 1. Settings

These are the only values that normally need editing.

* `INPUT_FILE`: name of the uploaded Excel file.
* `OUTPUT_FILE`: name of the Excel file the notebook writes.
* `SHIFTS_BY_DAY`: which shifts exist on each weekday. Public holidays use the Sunday shifts (S1, S2, S3). The number of X shifts on a day can be changed in the input Excel (section 2).
* `FORBIDDEN_PAIRS`: shift pairs on consecutive days that would leave less than 11 hours of rest.
* `MAX_SHIFTS_PER_WEEK`: the most shifts one employee can get in one Monday–Sunday week.
* `MAX_SOLVE_SECONDS`: how long the solver may search for a schedule.
* `UNFILLED_COST`: how bad it is to leave a shift unfilled. Other shifts cost 10 and X shifts 1, so when the solver must choose, it leaves an X shift unfilled rather than any other shift.

In [ ]:
!pip install ortools

from ortools.sat.python import cp_model
import pandas as pd
import re

INPUT_FILE = "example_input.xlsx"
OUTPUT_FILE = "vuorot_ratkaisu.xlsx"

# Shifts that exist on each weekday (0 = Monday ... 6 = Sunday).
# Public holidays always get the Sunday shifts, whatever the weekday.
SHIFTS_BY_DAY = {
    0: ["X", "I1", "I2", "I3"],      # Monday
    1: ["I1", "I2", "I3"],           # Tuesday
    2: ["I1", "I2", "I3"],           # Wednesday
    3: ["I1", "I2", "I3"],           # Thursday
    4: ["I1", "I2", "I3"],           # Friday
    5: ["L1", "L2", "L3", "L4"],     # Saturday
    6: ["S1", "S2", "S3"],           # Sunday or public holiday
}
HOLIDAY = 6  # a holiday uses the same shifts as Sunday

# 11 h rest rule: (shift today, shift tomorrow) that one person may not work.
FORBIDDEN_PAIRS = [
    ("I3", "L1"),
    ("I3", "L2"),
    ("I2", "L1"),
    ("L4", "S1"),
    ("I1", "X"),  # summer
    ("I2", "X"),  # summer
    ("I3", "X"),  # summer
]

MAX_SHIFTS_PER_WEEK = 2
MAX_SOLVE_SECONDS = 10

# Cost of one unfilled place in the objective. Higher = filled first.
UNFILLED_COST = {"X": 1}   # X shifts
DEFAULT_UNFILLED_COST = 10 # every other shift

# 2. Read the input Excel

**Calendar (first sheet).** One row per day with the columns `pv` (weekday), `pvm` (date), an "x" in the third column on public holidays, and optionally the number of X shifts in the fourth column. From it we get:

* `num_days`: the length of the planning period
* `day_type[d]`: the weekday number of day `d` (0–6), or 6 if `d` is a holiday
* `shifts_on[d]`: the list of shifts that exist on day `d`
* `demand[(d, s)]`: how many employees shift `s` on day `d` needs. This is 1 for every shift except X, which can need several.

**X shifts.** If the fourth column has a number, day `d` gets that many X shifts (also on holidays and other weekdays). If it is empty, the default from `SHIFTS_BY_DAY` applies: 1 X shift on Mondays, none on other days. A `0` removes the X shift.

**Employees (all other sheets).** Each sheet is one employee, and the sheet name is the employee's name. Adding or removing a sheet is all it takes to change the staff.

In [ ]:
sheets = pd.read_excel(INPUT_FILE, sheet_name=None)  # {sheet name: table}
sheet_names = list(sheets.keys())

calendar = sheets[sheet_names[0]]       # first sheet = holiday calendar
employees = sheet_names[1:]             # every other sheet = one employee

num_days = len(calendar)
all_days = range(num_days)
weekday_names = calendar.iloc[:, 0].astype(str).str.strip().tolist()  # "ma", "ti", ...
dates = calendar.iloc[:, 1].astype(str).str.strip().tolist()          # "31.3.", ...
holiday_marks = calendar.iloc[:, 2]

# The weekly rule counts Monday–Sunday weeks, so the period must be whole weeks.
if weekday_names[0].lower() != "ma" or num_days % 7 != 0:
    raise ValueError("Jakson pitää alkaa maanantaina ja koostua kokonaisista viikoista.")
num_weeks = num_days // 7

# Day type: the normal weekday, or the Sunday type on public holidays.
day_type = []
for d in all_days:
    is_holiday = str(holiday_marks.iloc[d]).strip().lower() == "x"
    day_type.append(HOLIDAY if is_holiday else d % 7)

# X shifts: the number in the fourth column, or the default from SHIFTS_BY_DAY if it is empty.
x_column = calendar.iloc[:, 3] if calendar.shape[1] > 3 else pd.Series([None] * num_days)

shifts_on = {}   # day -> list of shifts on that day
demand = {}      # (day, shift) -> number of employees needed
for d in all_days:
    default_shifts = SHIFTS_BY_DAY[day_type[d]]
    x_cell = x_column.iloc[d]
    if pd.isna(x_cell) or str(x_cell).strip() == "":
        x_count = 1 if "X" in default_shifts else 0
    else:
        x_count = int(x_cell)

    other_shifts = [s for s in default_shifts if s != "X"]
    shifts_on[d] = (["X"] if x_count > 0 else []) + other_shifts
    for s in other_shifts:
        demand[(d, s)] = 1
    if x_count > 0:
        demand[(d, "X")] = x_count

print(f"{len(employees)} työntekijää, {num_days} päivää ({num_weeks} viikkoa)")
print("X-vuorot:", {dates[d]: demand[(d, "X")] for d in all_days if (d, "X") in demand})
print("Pyhäpäivät:", [dates[d] for d in all_days if str(holiday_marks.iloc[d]).strip().lower() == "x"])

# 3. Read the shift requests

Each employee's `Vuorotoive` cell is turned into the set of shifts that person can work that day, `allowed[(e, d)]`. The rules, checked in this order:

1. Text contains **"vapaatoive"** → no shifts.
2. Text contains **"kaikki käy"** → all of the day's shifts (any extra text, like "mieluiten I3", is ignored).
3. Text contains **shift codes** (X, I1–I3, L1–L4, S1–S3, upper or lower case) → only those codes. A code that does not exist on that day is dropped, so "I1 ja I2 käy" on a Saturday means no shifts.
4. Anything else, including an **empty cell** → no shifts.

Every cell handled by rule 4, and every cell where rule 3 dropped a code, is printed so the team lead can check it.

In [ ]:
SHIFT_CODE = re.compile(r"\b(X|I[1-3]|L[1-4]|S[1-3])\b", re.IGNORECASE)

allowed = {}        # (employee, day) -> list of shifts the employee can work
to_check = []       # cells the team lead should look at

for e in employees:
    sheet = sheets[e]
    if len(sheet) < num_days:
        raise ValueError(f"Välilehdellä '{e}' on vain {len(sheet)} päivää, tarvitaan {num_days}.")
    requests = sheet["Vuorotoive"]

    for d in all_days:
        raw = requests.iloc[d]
        text = "" if pd.isna(raw) else str(raw).strip()
        lower = text.lower()

        if "vapaatoive" in lower:
            allowed[(e, d)] = []
        elif "kaikki käy" in lower:
            allowed[(e, d)] = list(shifts_on[d])
        else:
            codes = [c.upper() for c in SHIFT_CODE.findall(text)]
            valid = [c for c in codes if c in shifts_on[d]]
            allowed[(e, d)] = valid
            if len(valid) < len(codes) or not codes:
                to_check.append((e, dates[d], text if text else "(tyhjä)", valid))

print(f"Tarkistettavat solut ({len(to_check)} kpl), tulkittu vain alla näkyvät vuorot sallituiksi:")
for e, date, text, valid in to_check:
    print(f"  {e:10} {date:7} '{text}' -> {valid if valid else 'ei vuoroa'}")

# 4. Create the model and its variables

* `works[(e, d, s)]` is 1 if employee `e` works shift `s` on day `d`, otherwise 0. It is only created for shifts the employee's request allows, so the solver can never give anyone a shift they did not ask for.
* `unfilled[(d, s)]` is the number of places in shift `s` on day `d` that nobody works. It is 0 or 1 for normal shifts and can be up to `demand[(d, s)]` for X shifts.

In [ ]:
model = cp_model.CpModel()

works = {}
for (e, d), shift_list in allowed.items():
    for s in shift_list:
        works[(e, d, s)] = model.NewBoolVar(f"works_{e}_{d}_{s}")

unfilled = {}
for d in all_days:
    for s in shifts_on[d]:
        unfilled[(d, s)] = model.NewIntVar(0, demand[(d, s)], f"unfilled_{d}_{s}")

def shifts_of(e, days):
    # All works-variables of employee e on the given days.
    return [v for (emp, d, s), v in works.items() if emp == e and d in days]

# 5. Constraints

All four are hard rules: the schedule always follows them.

1. **Each shift gets the number of employees it needs** (1, or the X count), and any missing places are counted as unfilled.
2. **At most one shift per employee per day.**
3. **At most `MAX_SHIFTS_PER_WEEK` shifts per employee per Monday–Sunday week.**
4. **11 hours of rest:** nobody works a forbidden pair on two consecutive days.

Shift requests need no constraint of their own, because the variables in section 4 only exist for allowed shifts.

In [ ]:
# 1. Each shift: employees working it + unfilled places = places needed (1, or the X count).
for d in all_days:
    for s in shifts_on[d]:
        assigned = [works[(e, d, s)] for e in employees if (e, d, s) in works]
        model.Add(sum(assigned) + unfilled[(d, s)] == demand[(d, s)])

# 2. At most one shift per employee per day.
for e in employees:
    for d in all_days:
        model.Add(sum(shifts_of(e, [d])) <= 1)

# 3. At most MAX_SHIFTS_PER_WEEK shifts per employee per week.
for e in employees:
    for w in range(num_weeks):
        week_days = range(w * 7, w * 7 + 7)
        model.Add(sum(shifts_of(e, week_days)) <= MAX_SHIFTS_PER_WEEK)

# 4. Forbidden pairs on consecutive days (11 h rest).
for e in employees:
    for d in range(num_days - 1):
        for today, tomorrow in FORBIDDEN_PAIRS:
            if (e, d, today) in works and (e, d + 1, tomorrow) in works:
                model.Add(works[(e, d, today)] + works[(e, d + 1, tomorrow)] <= 1)

# 6. Objective and solving

The solver minimises the total cost of unfilled shifts: each unfilled place costs `UNFILLED_COST` (1 for X shifts) or `DEFAULT_UNFILLED_COST` (10 for all others). So if the solver has to choose between filling I1 and filling X, it fills I1. Among shifts with the same cost it still leaves as few unfilled as possible.

Because one other shift costs as much as ten X shifts, the solver would leave up to nine X places unfilled to fill one other shift. In practice the trade is almost always one for one.

Status codes:
* **OPTIMAL**: proven best schedule (this should be the normal result)
* **FEASIBLE**: a valid schedule, but not proven best within the time limit
* **INFEASIBLE**: no valid schedule exists

In [ ]:
model.Minimize(sum(UNFILLED_COST.get(s, DEFAULT_UNFILLED_COST) * var
                   for (d, s), var in unfilled.items()))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = MAX_SOLVE_SECONDS
status = solver.Solve(model)

print("Status:", solver.StatusName(status))
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Täyttämättömät vuorot:", sum(solver.Value(v) for v in unfilled.values()))

# 7. Output Excel

One sheet in the same layout as the old model: one row per employee, one column per day, and the shift code in each cell (empty = no shift).

Below the schedule is the list of **unfilled shifts** (day and shift code), coloured red. An X shift with two missing places appears twice. The team lead fills these manually.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    day_headers = [f"{weekday_names[d].capitalize()} {dates[d]}" for d in all_days]  # e.g. "Ma 31.3."

    # Schedule: employee -> the shift code worked on each day ("" = day off).
    schedule = pd.DataFrame("", index=employees, columns=day_headers)
    for (e, d, s), var in works.items():
        if solver.Value(var):
            schedule.loc[e, day_headers[d]] = s

    # Unfilled places: one row per missing place (an X shift missing 2 people gives 2 rows).
    missing = [(day_headers[d], s)
               for (d, s), var in unfilled.items()
               for _ in range(solver.Value(var))]
    missing_table = pd.DataFrame(missing, columns=["Päivä", "Vuoro"])

    # Write the schedule at the top and the unfilled list two rows below it.
    missing_title_row = len(employees) + 3          # Excel row number of the title
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        schedule.to_excel(writer, sheet_name="Vuorot")
        missing_table.to_excel(writer, sheet_name="Vuorot", startrow=missing_title_row, index=False)
        writer.sheets["Vuorot"].cell(row=missing_title_row, column=1, value="Täyttämättömät vuorot:")

    # Colour the unfilled list red (its header row and every row below it).
    wb = load_workbook(OUTPUT_FILE)
    ws = wb["Vuorot"]
    red = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")
    first_row = missing_title_row + 1
    for row in ws.iter_rows(min_row=first_row, max_row=first_row + len(missing), max_col=2):
        for cell in row:
            cell.fill = red
    wb.save(OUTPUT_FILE)

    print(f"Vuorolista tallennettu: {OUTPUT_FILE} ({len(missing)} täyttämätöntä vuoroa)")
else:
    print("Vuorolistaa ei voitu muodostaa.")